# Garmin Connect — data fetch & format

Connect to Garmin Connect, fetch each dataset, and return formatted results.

Set `GARMIN_EMAIL` and `GARMIN_PASSWORD` in the repo-root `.env` file.

## Setup

In [ ]:
import os
import sys
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Any

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "garmin-connect" else NOTEBOOK_DIR

env_path = REPO_ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

try:
    from garminconnect import Garmin
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "garminconnect"])
    from garminconnect import Garmin

GARMIN_EMAIL = os.environ.get("GARMIN_EMAIL")
GARMIN_PASSWORD = os.environ.get("GARMIN_PASSWORD")

## Shared helpers

In [ ]:
def normalize_hr_zones(raw: Any) -> dict[int, float]:
    zones: dict[int, float] = {}
    if not isinstance(raw, list):
        return zones
    for entry in raw:
        if not isinstance(entry, dict) or "zoneNumber" not in entry:
            continue
        zones[int(entry["zoneNumber"])] = float(entry.get("secsInZone") or 0)
    return zones


def zones_to_minute_columns(zones: dict[int, float]) -> list[float | None]:
    if not zones:
        return [None] * 5
    return [round(zones.get(zone, 0) / 60, 2) for zone in range(1, 6)]


def running_activities(client: Garmin, start_date: str, end_date: str) -> list[dict[str, Any]]:
    activities = client.get_activities_by_date(start_date, end_date)
    return [
        a for a in activities
        if a.get("activityType", {}).get("typeKey") == "running"
    ]


def fetch_hr_zones(client: Garmin, activity_id: Any) -> dict[int, float]:
    return normalize_hr_zones(client.get_activity_hr_in_timezones(str(activity_id)))


def _primary_device_entry(device_map: dict[str, Any]) -> dict[str, Any]:
    if not isinstance(device_map, dict):
        return {}
    for entry in device_map.values():
        if isinstance(entry, dict) and entry.get("primaryTrainingDevice"):
            return entry
    for entry in device_map.values():
        if isinstance(entry, dict):
            return entry
    return {}


def parse_training_status(raw: dict[str, Any]) -> dict[str, Any]:
    status = _primary_device_entry(
        (raw.get("mostRecentTrainingStatus") or {}).get("latestTrainingStatusData") or {}
    )
    load = status.get("acuteTrainingLoadDTO") or {}
    balance = _primary_device_entry(
        (raw.get("mostRecentTrainingLoadBalance") or {}).get("metricsTrainingLoadBalanceDTOMap") or {}
    )
    return {
        "calendar_date": status.get("calendarDate"),
        "training_status": status.get("trainingStatusFeedbackPhrase"),
        "fitness_trend": status.get("fitnessTrend"),
        "acute_load": load.get("dailyTrainingLoadAcute"),
        "chronic_load": load.get("dailyTrainingLoadChronic"),
        "acwr": load.get("dailyAcuteChronicWorkloadRatio"),
        "acwr_status": load.get("acwrStatus"),
        "chronic_load_min": load.get("minTrainingLoadChronic"),
        "chronic_load_max": load.get("maxTrainingLoadChronic"),
        "load_balance_feedback": balance.get("trainingBalanceFeedbackPhrase"),
        "monthly_aerobic_low": round(balance.get("monthlyLoadAerobicLow", 0) or 0, 1),
        "monthly_aerobic_high": round(balance.get("monthlyLoadAerobicHigh", 0) or 0, 1),
        "monthly_anaerobic": round(balance.get("monthlyLoadAnaerobic", 0) or 0, 1),
    }

## Connect

In [ ]:
if not GARMIN_EMAIL or not GARMIN_PASSWORD:
    raise RuntimeError("Set GARMIN_EMAIL and GARMIN_PASSWORD in the repo-root .env file.")

client = Garmin(email=GARMIN_EMAIL, password=GARMIN_PASSWORD)
client.login()

## Profile

In [ ]:
def format_profile(client: Garmin) -> dict[str, Any]:
    profile = (client.get_user_profile() or {}).get("userData", {})
    training = parse_training_status(client.get_training_status(date.today().isoformat()))
    return {
        "weight": round(profile.get("weight", 0) / 1000, 2),
        "height": profile.get("height"),
        "birthDate": profile.get("birthDate"),
        "vo2MaxRunning": profile.get("vo2MaxRunning"),
        "lactateThresholdSpeed": round(
            (profile.get("lactateThresholdSpeed", 0) * 1000 / 60), 2
        ),
        "lactateThresholdHeartRate": profile.get("lactateThresholdHeartRate"),
        "availableTrainingDays": profile.get("availableTrainingDays", []),
        "preferredLongTrainingDays": profile.get("preferredLongTrainingDays", []),
        "training_status": training.get("training_status"),
        "acute_load": training.get("acute_load"),
        "chronic_load": training.get("chronic_load"),
        "acwr": training.get("acwr"),
        "acwr_status": training.get("acwr_status"),
    }

In [ ]:
profile = format_profile(client)
profile

## Activities

In [ ]:
ACTIVITY_HEADERS = [
    "name", "type", "distance_km", "duration_min", "avg_pace_min_per_km",
    "max_pace_min_per_km", "max_hr", "avg_hr", "aerobic_te", "anaerobic_te",
    "training_effect", "zone_1_min", "zone_2_min", "zone_3_min", "zone_4_min", "zone_5_min",
]


def format_activity_row(activity: dict[str, Any], zones: dict[int, float]) -> list[Any]:
    avg_speed = activity.get("averageSpeed")
    max_speed = activity.get("maxSpeed")
    return [
        activity.get("activityName"),
        activity.get("activityType", {}).get("typeKey"),
        round((activity.get("distance", 0) or 0) / 1000, 2),
        round((activity.get("movingDuration", 0) or 0) / 60, 2),
        round((1000 / avg_speed) / 60, 2) if avg_speed else None,
        round((1000 / max_speed) / 60, 2) if max_speed else None,
        activity.get("maxHR"),
        activity.get("averageHR"),
        round(activity.get("aerobicTrainingEffect", 0) or 0, 2),
        round(activity.get("anaerobicTrainingEffect", 0) or 0, 2),
        activity.get("trainingEffectLabel"),
        *zones_to_minute_columns(zones),
    ]


def format_activities(client: Garmin, days_back: int = 7) -> dict[str, Any]:
    today = date.today()
    activities = running_activities(
        client,
        (today - timedelta(days=days_back)).isoformat(),
        today.isoformat(),
    )
    rows = []
    for activity in activities:
        zones = fetch_hr_zones(client, activity.get("activityId"))
        rows.append(format_activity_row(activity, zones))
    return {"Garmin Activities": {"Headers": ACTIVITY_HEADERS, "Rows": rows}}

In [ ]:
activities = format_activities(client, days_back=7)
activities

## Race predictions

In [ ]:
def format_race_predictions(client: Garmin) -> dict[str, Any]:
    raw = client.get_race_predictions()
    return {
        "Garmin Race Predictions": {
            "Headers": ["5k_min", "10k_min", "half_marathon_h", "marathon_h"],
            "Rows": [[
                round((raw.get("time5K", 0) or 0) / 60, 2),
                round((raw.get("time10K", 0) or 0) / 60, 2),
                round((raw.get("timeHalfMarathon", 0) or 0) / 3600, 2),
                round((raw.get("timeMarathon", 0) or 0) / 3600, 2),
            ]],
        }
    }

In [ ]:
race_predictions = format_race_predictions(client)
race_predictions

## Events

In [ ]:
def format_events(client: Garmin, months_ahead: int = 12) -> dict[str, Any]:
    today = date.today()
    start_month = today.replace(day=1)
    event_rows = []
    seen_titles: set[str] = set()

    for offset in range(months_ahead):
        year = start_month.year + (start_month.month - 1 + offset) // 12
        month = (start_month.month - 1 + offset) % 12 + 1
        calendar = client.get_scheduled_workouts(year=year, month=month)
        if not isinstance(calendar, dict):
            continue
        for event in calendar.get("calendarItems", []):
            if not isinstance(event, dict) or event.get("itemType") != "event":
                continue
            event_date_str = event.get("date")
            if not event_date_str:
                continue
            try:
                event_date = date.fromisoformat(event_date_str)
            except ValueError:
                continue
            if event_date <= today:
                continue
            title = (event.get("title") or "").strip().lower()
            if title in seen_titles:
                continue
            seen_titles.add(title)
            target = event.get("completionTarget") or {}
            event_rows.append([
                event.get("title"), event.get("itemType"), event_date_str,
                target.get("value"), target.get("unit"), target.get("unitType"),
            ])

    return {
        "Garmin Events": {
            "Headers": ["title", "item_type", "date", "target_value", "target_unit", "target_unit_type"],
            "Rows": event_rows,
        }
    }

In [ ]:
events = format_events(client, months_ahead=12)
events

## Weekly stats

In [ ]:
WEEKLY_STATS_HEADERS = [
    "week_start", "week_end", "distance_km", "zone_1_min", "zone_2_min",
    "zone_3_min", "zone_4_min", "zone_5_min", "easy_min", "hard_min", "easy_pct",
]


def activity_date(activity: dict[str, Any]) -> date | None:
    start_time = activity.get("startTimeLocal", "")
    if not start_time:
        return None
    try:
        return date.fromisoformat(start_time[:10])
    except ValueError:
        return None


def easy_hard_from_zones(zones: dict[int, float]) -> tuple[float, float, float | None]:
    easy_secs = zones.get(1, 0) + zones.get(2, 0)
    hard_secs = zones.get(3, 0) + zones.get(4, 0) + zones.get(5, 0)
    total_secs = easy_secs + hard_secs
    easy_pct = round(100 * easy_secs / total_secs, 1) if total_secs else None
    return round(easy_secs / 60, 2), round(hard_secs / 60, 2), easy_pct


def weekly_stats_rows(
    activities: list[dict[str, Any]],
    activity_zones: dict[Any, dict[int, float]],
    today: date,
    num_blocks: int = 4,
) -> list[list[Any]]:
    rows: list[list[Any]] = []
    for block_index in range(num_blocks - 1, -1, -1):
        block_end = today - timedelta(days=block_index * 7)
        block_start = block_end - timedelta(days=6)
        block_distance_km = 0.0
        block_zones = {zone: 0.0 for zone in range(1, 6)}
        for activity in activities:
            act_date = activity_date(activity)
            if act_date is None or not (block_start <= act_date <= block_end):
                continue
            block_distance_km += (activity.get("distance", 0) or 0) / 1000
            zones = activity_zones.get(activity.get("activityId"), {})
            for zone, secs in zones.items():
                if 1 <= zone <= 5:
                    block_zones[zone] += secs
        easy_min, hard_min, easy_pct = easy_hard_from_zones(block_zones)
        rows.append([
            block_start.isoformat(), block_end.isoformat(), round(block_distance_km, 2),
            *[round(block_zones[zone] / 60, 2) for zone in range(1, 6)],
            easy_min, hard_min, easy_pct,
        ])
    return rows


def format_weekly_stats(client: Garmin, weeks: int = 4) -> dict[str, Any]:
    today = date.today()
    activities = running_activities(
        client,
        (today - timedelta(days=weeks * 7 - 1)).isoformat(),
        today.isoformat(),
    )
    activity_zones = {
        a["activityId"]: fetch_hr_zones(client, a["activityId"])
        for a in activities if a.get("activityId") is not None
    }
    return {
        "Garmin Weekly Stats": {
            "Headers": WEEKLY_STATS_HEADERS,
            "Rows": weekly_stats_rows(activities, activity_zones, today, num_blocks=weeks),
        }
    }

In [ ]:
weekly_stats = format_weekly_stats(client, weeks=4)
weekly_stats

## Training load — coach reference

Notes for system prompt rules. How to interpret `get_training_status` fields and decide whether training is building, maintaining, or backing off.

### Key fields

| Field | Garmin API name | Meaning |
|---|---|---|
| `acute_load` | `dailyTrainingLoadAcute` | Recent 7-day training strain (short-term fatigue) |
| `chronic_load` | `dailyTrainingLoadChronic` | Longer-term fitness base (~28-day rolling load) |
| `acwr` | `dailyAcuteChronicWorkloadRatio` | Acute ÷ chronic — the key ratio for load decisions |
| `training_status` | `trainingStatusFeedbackPhrase` | Garmin label, e.g. `PRODUCTIVE_3`, `MAINTAINING_4` |
| `acwr_status` | `acwrStatus` | Garmin's assessment, e.g. `OPTIMAL` |

Load decays each day without new training (Garmin uses exponential decay, similar to Banister/EWMA). Chronic load moves slowly; acute load reacts quickly to recent sessions.

### ACWR is a point-in-time snapshot

`acwr` from `get_training_status(today)` answers: **right now, how does recent strain compare to my fitness base?**

- **ACWR = 0.8** → recent load is 80% of chronic base → under-loading (recovery week, light week, or detraining)
- **ACWR ≈ 1.0** → recent load matches base → maintaining
- **ACWR 1.1–1.2** → acute above chronic → base should climb over coming weeks if sustained
- **ACWR > 1.3** → spike → injury risk; hold or reduce volume

Do **not** average ACWR across weeks — it blurs the signal. ACWR already embeds time decay; averaging it adds little value.

### What to track instead of average ACWR

| Metric | Use |
|---|---|
| **Current ACWR** | Immediate state: building, maintaining, or backing off |
| **Chronic load at week-end** | Is the fitness base actually rising over time? |
| **Week-over-week chronic delta %** | Did base load increase sustainably (~10% per week) or not? |

Km and HR zones alone do not show whether the body is absorbing the work. Chronic load trend + current ACWR together answer that.

### Progressive overload heuristic

Without new training, load decays daily. To **maintain** chronic load, keep feeding in similar weekly load. To **increase** chronic load (build fitness), acute must exceed chronic for a sustained period.

```
if ACWR < 1.0   → not building; chronic base flat or falling
if ACWR 1.0–1.3 → progressive overload zone (1.1–1.2 is the sweet spot)
if ACWR > 1.3   → spike; hold or reduce volume
```

A ~10% weekly increase in chronic load maps to keeping ACWR slightly above 1.0 without spiking past ~1.3. If km went up but chronic load is flat, intensity may have increased without the base absorbing it. If chronic is rising and ACWR stays ~1.1–1.2, training is building sustainably.

### Combining with weekly stats

- **Weekly stats** (km, HR zones) → what was done
- **Training load** (acute, chronic, ACWR) → what effect it had on the body

Use both when evaluating past weeks and planning increases.

## Training periodization — coach reference

Notes for system prompt rules. Volume progression, recovery cycles, event taper, and intensity split per week type.

### Week-type decision (priority order)

No stored plan start. Count weeks **forward** from the first week in the lookback window (`week_number` 1, 2, 3…). Use the **next event** (first upcoming from events list) for taper/race.

```
1. Race week          (0 weeks to event at week_end)
2. Taper final        (1 week to event)   → volume × 0.64 of peak
3. Taper first        (2 weeks to event)  → volume × 0.80 of peak
4. Recovery           (week_number % 4 == 0) → volume × 0.80 of previous week
5. Build              (everything else)   → volume up to +10% vs previous week
```

**Taper overrides recovery.**

### Formula

```python
def weeks_until_event(from_date: date, event_date: date) -> int:
    return max(0, (event_date - from_date).days // 7)


def classify_week_type(week_number: int, weeks_until: int | None) -> str:
    if weeks_until is not None:
        if weeks_until == 0:
            return "race"
        if weeks_until == 1:
            return "taper_final"
        if weeks_until == 2:
            return "taper_first"
    if week_number % 4 == 0:
        return "recovery"
    return "build"
```

### Volume & intensity by week type

| Week type | Volume | Easy % | Hard % |
|---|---|---|---|
| build | up to +10% vs prev week | 80 | 20 |
| recovery | 80% of prev week | 90 | 10 |
| taper_first | 80% of peak build week | 80 | 20 |
| taper_final | 64% of peak build week | 85 | 15 |
| race | ~30% shakeout | 90 | 10 |

### Agent rules

- Compare `distance_km` vs `volume_target_km` and `easy_pct` vs `target_easy_pct`.
- Use `acwr` — if > 1.3, do not increase volume even on build weeks.
- The row with `is_plan=true` is the week to schedule sessions for.


## Training load (current)

From `get_training_status(date)`. Acute load = recent 7-day strain. Chronic load = longer-term fitness base (~28 days). ACWR = acute / chronic.

In [ ]:
def format_training_load(client: Garmin, cdate: str | None = None) -> dict[str, Any]:
    cdate = cdate or date.today().isoformat()
    return parse_training_status(client.get_training_status(cdate))

In [ ]:
training_load = format_training_load(client)
training_load

## Training load (weekly history)

Samples `get_training_status` at each week-end to track how load evolved, plus rolling averages the agent can use to judge safe increases.

In [ ]:
TRAINING_LOAD_HEADERS = [
    "week_start", "week_end", "acute_load", "chronic_load", "acwr",
    "training_status", "acwr_status",
]


def format_training_load_weeks(client: Garmin, weeks: int = 4) -> dict[str, Any]:
    today = date.today()
    rows: list[list[Any]] = []
    acute_values: list[float] = []
    chronic_values: list[float] = []

    for block_index in range(weeks - 1, -1, -1):
        block_end = today - timedelta(days=block_index * 7)
        block_start = block_end - timedelta(days=6)
        parsed = parse_training_status(
            client.get_training_status(block_end.isoformat())
        )
        time.sleep(0.2)

        acute = parsed.get("acute_load")
        chronic = parsed.get("chronic_load")
        if acute is not None:
            acute_values.append(float(acute))
        if chronic is not None:
            chronic_values.append(float(chronic))

        rows.append([
            block_start.isoformat(),
            block_end.isoformat(),
            acute,
            chronic,
            parsed.get("acwr"),
            parsed.get("training_status"),
            parsed.get("acwr_status"),
        ])

    return {
        "Garmin Training Load": {
            "Headers": TRAINING_LOAD_HEADERS,
            "Rows": rows,
            "Averages": {
                "avg_acute_load": round(sum(acute_values) / len(acute_values), 1) if acute_values else None,
                "avg_chronic_load": round(sum(chronic_values) / len(chronic_values), 1) if chronic_values else None,
            },
        }
    }

In [ ]:
training_load_weeks = format_training_load_weeks(client, weeks=4)
training_load_weeks

## Training plan (`get_training_plan`)

Array of week objects — **4 past weeks + 1 upcoming week**. Replaces `weekly_stats` and `week_context` in the weekly report.

Event context comes from `events` separately — not duplicated here.

```python
training_plan = [
  {
    "week_start": "...",
    "week_end": "...",
    "week_description": "past_week" | "current_week" | "upcoming_week",
    "week_type": "build" | "recovery" | "taper_first" | ...,
    "actuals": { ... },   # Garmin data (null on upcoming_week)
    "target": { ... },    # Calculated plan for that week
  },
]
```


### How to interpret the training plan

Use this block together with `events` (next race) and `profile` (athlete context). The agent's job is to **review the past**, **assess the present**, and **plan the upcoming week**.

#### Week descriptions

| `week_description` | Meaning | Agent focus |
|---|---|---|
| `past_week` | Completed Mon–Sun | Compare `actuals` vs `target` — did the athlete follow the plan? |
| `current_week` | In progress (ends today) | `actuals` are partial; use `target` for what remains this week |
| `upcoming_week` | Next Mon–Sun | `actuals` are empty; **schedule sessions from `target`** |

#### `week_type` (calculated, never stored)

Derived from week number (counting forward from oldest week) and weeks until the next event:

- **build** — up to +10% distance vs previous week; 80/20 easy/hard
- **recovery** — every 4th week; 80% distance; 90/10 easy/hard
- **taper_first** / **taper_final** — 2 and 1 weeks before event; reduced volume
- **race** — event week; minimal shakeout volume

Taper overrides recovery when both would apply.

#### Reading `actuals`

| Field | What it tells you |
|---|---|
| `distance_km` | Weekly running volume |
| `zone_1_min` … `zone_5_min` | Time in each HR zone |
| `easy_min` / `hard_min` | Z1–Z2 vs Z3–Z5 time |
| `easy_pct` / `hard_pct` | Intensity distribution (polarized target ~80/20) |
| `acute_load` | Recent 7-day training strain |
| `chronic_load` | Longer-term fitness base (~28 days) |
| `acwr` | Acute ÷ chronic — point-in-time load balance |

**ACWR guide:** `< 1.0` = under-loading · `1.0–1.3` = building safely · `> 1.3` = too much, hold volume.

#### Reading `target`

What that week **should** have looked like (past/current) or **should look like** (upcoming):

| Field | Notes |
|---|---|
| `distance_km` | Planned weekly km from periodization rules |
| `easy_pct` / `hard_pct` | Planned intensity split for that week type |
| `acute_load` / `chronic_load` / `acwr` | Always `null` — Garmin derives these from execution, not prescriptive |

#### Agent workflow

1. **Review past weeks** — flag weeks where `actuals.distance_km` diverged >10% from `target`, or easy/hard split was off.
2. **Check load** — if latest `actuals.acwr` > 1.3, do not increase volume on `upcoming_week` even if `week_type` is build.
3. **Plan upcoming week** — use the `upcoming_week` entry: respect `week_type`, hit `target.distance_km`, distribute sessions to match `target.easy_pct` / `target.hard_pct`.
4. **Cross-reference `events`** — confirm taper/recovery timing against the race date from the events list.

#### Example questions the plan answers

- "Did last week match the recovery plan?" → compare last `past_week` actuals vs target
- "How much should I run next week?" → `upcoming_week.target.distance_km`
- "Can I increase load?" → latest `acwr` + `upcoming_week.week_type`
- "Should next week include intervals?" → `upcoming_week.week_type` and `target.hard_pct`

In [ ]:
WEEK_TYPE_SPECS = {
    "build": {"multiplier": 1.10, "easy_pct": 80, "hard_pct": 20, "ref": "prev"},
    "recovery": {"multiplier": 0.80, "easy_pct": 90, "hard_pct": 10, "ref": "prev"},
    "taper_first": {"multiplier": 0.80, "easy_pct": 80, "hard_pct": 20, "ref": "peak"},
    "taper_final": {"multiplier": 0.64, "easy_pct": 85, "hard_pct": 15, "ref": "peak"},
    "race": {"multiplier": 0.30, "easy_pct": 90, "hard_pct": 10, "ref": "prev"},
}


def weeks_until_event(from_date: date, event_date: date) -> int:
    return max(0, (event_date - from_date).days // 7)


def classify_week_type(week_number: int, weeks_until: int | None) -> str:
    if weeks_until is not None:
        if weeks_until == 0:
            return "race"
        if weeks_until == 1:
            return "taper_final"
        if weeks_until == 2:
            return "taper_first"
    if week_number % 4 == 0:
        return "recovery"
    return "build"


def _event_date_from_events(events: dict[str, Any]) -> date | None:
    today = date.today()
    upcoming = []
    for row in events.get("Garmin Events", {}).get("Rows", []):
        try:
            event_date = date.fromisoformat(row[2])
        except (ValueError, IndexError):
            continue
        if event_date > today:
            upcoming.append(event_date)
    return min(upcoming) if upcoming else None


def calc_distance_target(week_type: str, prev_km: float | None, peak_km: float) -> float | None:
    spec = WEEK_TYPE_SPECS[week_type]
    if week_type == "build":
        if not prev_km:
            return None
        return round(prev_km * spec["multiplier"], 1)
    base = peak_km if spec["ref"] == "peak" else (prev_km or 0)
    if base <= 0:
        return None
    return round(base * spec["multiplier"], 1)


def build_target(week_type: str, prev_km: float | None, peak_km: float) -> dict[str, Any]:
    spec = WEEK_TYPE_SPECS[week_type]
    return {
        "distance_km": calc_distance_target(week_type, prev_km, peak_km),
        "easy_pct": spec["easy_pct"],
        "hard_pct": spec["hard_pct"],
        "acute_load": None,
        "chronic_load": None,
        "acwr": None,
    }


def actuals_from_stat_row(row: list[Any], load: dict[str, Any]) -> dict[str, Any]:
    easy_pct = row[10]
    return {
        "distance_km": row[2],
        "zone_1_min": row[3],
        "zone_2_min": row[4],
        "zone_3_min": row[5],
        "zone_4_min": row[6],
        "zone_5_min": row[7],
        "easy_min": row[8],
        "hard_min": row[9],
        "easy_pct": easy_pct,
        "hard_pct": round(100 - easy_pct, 1) if easy_pct is not None else None,
        "acute_load": load.get("acute_load"),
        "chronic_load": load.get("chronic_load"),
        "acwr": load.get("acwr"),
    }


def empty_actuals() -> dict[str, Any]:
    return {
        "distance_km": None,
        "zone_1_min": None,
        "zone_2_min": None,
        "zone_3_min": None,
        "zone_4_min": None,
        "zone_5_min": None,
        "easy_min": None,
        "hard_min": None,
        "easy_pct": None,
        "hard_pct": None,
        "acute_load": None,
        "chronic_load": None,
        "acwr": None,
    }


def next_week_after(week_end: date) -> tuple[date, date]:
    start = week_end + timedelta(days=1)
    while start.weekday() != 0:
        start += timedelta(days=1)
    return start, start + timedelta(days=6)


def format_training_plan(
    client: Garmin,
    events: dict[str, Any],
    weeks: int = 4,
) -> list[dict[str, Any]]:
    event_date = _event_date_from_events(events)
    stat_rows = format_weekly_stats(client, weeks=weeks)["Garmin Weekly Stats"]["Rows"]

    week_types = []
    for i, row in enumerate(stat_rows):
        week_end = date.fromisoformat(row[1])
        w_until = weeks_until_event(week_end, event_date) if event_date else None
        week_types.append(classify_week_type(i + 1, w_until))

    peak_km = max(
        (float(row[2] or 0) for row, wt in zip(stat_rows, week_types) if wt == "build"),
        default=max((float(row[2] or 0) for row in stat_rows), default=0),
    )

    training_plan: list[dict[str, Any]] = []
    prev_km: float | None = None

    for i, row in enumerate(stat_rows):
        week_start = date.fromisoformat(row[0])
        week_end = date.fromisoformat(row[1])
        week_type = week_types[i]
        is_current = i == len(stat_rows) - 1
        description = "current_week" if is_current else "past_week"

        load = parse_training_status(client.get_training_status(week_end.isoformat()))
        if not is_current:
            time.sleep(0.2)

        training_plan.append({
            "week_start": week_start.isoformat(),
            "week_end": week_end.isoformat(),
            "week_description": description,
            "week_type": week_type,
            "actuals": actuals_from_stat_row(row, load),
            "target": build_target(week_type, prev_km, peak_km),
        })
        prev_km = float(row[2] or 0) or prev_km

    plan_start, plan_end = next_week_after(date.fromisoformat(stat_rows[-1][1]))
    plan_week_num = len(stat_rows) + 1
    plan_w_until = weeks_until_event(plan_start, event_date) if event_date else None
    plan_type = classify_week_type(plan_week_num, plan_w_until)

    training_plan.append({
        "week_start": plan_start.isoformat(),
        "week_end": plan_end.isoformat(),
        "week_description": "upcoming_week",
        "week_type": plan_type,
        "actuals": empty_actuals(),
        "target": build_target(plan_type, prev_km, peak_km),
    })

    return training_plan


In [ ]:
training_plan = format_training_plan(client, events=events, weeks=4)
training_plan